# 02 — Baseline ResNet

Two training runs back to back:

**Run 1 — no imbalance correction.** Train and watch it fail. The confusion matrix will show the model collapsing toward `nv`.

**Run 2 — class-weighted loss.** Fix the imbalance problem, retrain, compare.

The point is to see the problem before seeing the fix.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

from data import get_dataloaders
from model import build_model
from train import train

DATA_DIR = '../data'
RESULTS_DIR = '../results'
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', DEVICE)

## Load data

The dataloader reads images from `data/train/`, `data/val/`, `data/test/` and returns them in batches.
It also tells us the class names in alphabetical order — that order matters when we read the confusion matrix later.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=32)

class_names = train_loader.dataset.classes
num_classes = len(class_names)
print('Classes:', class_names)
print('Batches per epoch (train):', len(train_loader))

# Majority-class baseline: accuracy a model gets by always predicting the most common class
from pathlib import Path
train_dir = Path(DATA_DIR) / 'train'
class_counts = {cls: len(list((train_dir / cls).glob('*.jpg'))) for cls in class_names}
total = sum(class_counts.values())
majority_class = max(class_counts, key=class_counts.get)
majority_baseline = class_counts[majority_class] / total
print(f"Majority class: '{majority_class}' ({majority_baseline:.1%} of train) — the baseline to beat")

## Run 1 — no imbalance correction

ResNet18 with its backbone frozen. Only the final classification layer trains.
No correction for the fact that `nv` is the majority class.
Prediction: val accuracy will look decent but the model will be mostly predicting `nv`.

*(Using ResNet18 here for speed — same concept as ResNet50, trains ~3× faster. ResNet50 is an option once the pipeline is confirmed working.)*

In [ ]:
model_v1 = build_model(num_classes=num_classes, backbone='resnet18', freeze_backbone=True)
history_v1 = train(model_v1, train_loader, val_loader, epochs=5, lr=1e-3, save_dir=RESULTS_DIR + '/v1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history_v1['train_loss']) + 1)

axes[0].plot(epochs, history_v1['train_loss'], label='train')
axes[0].plot(epochs, history_v1['val_loss'], label='val')
axes[0].set_title('Loss (no correction)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history_v1['val_acc'], color='steelblue')
axes[1].axhline(majority_baseline, linestyle='--', color='red', label=f'{majority_class}-always baseline ({majority_baseline:.0%})')
axes[1].set_title('Val accuracy (no correction)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.suptitle('Run 1 — no imbalance correction', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR + '/v1_training_curves.png', dpi=150)
plt.show()

### Confusion matrix — Run 1

The confusion matrix shows, for each true class (rows), what the model predicted (columns).
A perfect model has a bright diagonal and zeros everywhere else.
A model that always predicts `nv` will have the entire `nv` column lit up.

In [ ]:
model_v1.eval()
model_v1 = model_v1.to(DEVICE)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        preds = model_v1(images).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Confusion matrix — Run 1 (no correction)')
plt.tight_layout()
plt.savefig(RESULTS_DIR + '/v1_confusion_matrix.png', dpi=150)
plt.show()

---

## Run 2 — class-weighted loss

The fix: tell the loss function that mistakes on rare classes should cost more than mistakes on `nv`.
We compute a weight for each class that is inversely proportional to how often it appears.
`nv` gets a low weight (it's common, mistakes are cheap). `df` gets a high weight (it's rare, mistakes are expensive).

Nothing else changes — same model architecture, same data, same learning rate.

In [ ]:
# Compute class weights: inverse of class frequency
from pathlib import Path

train_dir = Path(DATA_DIR) / 'train'
counts = np.array([
    len(list((train_dir / cls).glob('*.jpg')))
    for cls in class_names
], dtype=np.float32)

weights = 1.0 / counts
weights = weights / weights.sum() * num_classes  # normalize so they average to 1
class_weights = torch.tensor(weights).to(DEVICE)

print('Class weights:')
for cls, w in zip(class_names, weights):
    print(f'  {cls:<8} {w:.4f}')

In [ ]:
import torch.nn as nn

model_v2 = build_model(num_classes=num_classes, backbone='resnet18', freeze_backbone=True)
model_v2 = model_v2.to(DEVICE)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_v2.parameters()), lr=1e-3
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

Path(RESULTS_DIR + '/v2').mkdir(parents=True, exist_ok=True)

history_v2 = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(5):
    model_v2.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_v2(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model_v2.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model_v2(images)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    val_loss /= len(val_loader)
    val_acc = correct / total

    history_v2['train_loss'].append(train_loss)
    history_v2['val_loss'].append(val_loss)
    history_v2['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/5 | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_v2.state_dict(), RESULTS_DIR + '/v2/best_model.pt')

print(f"\nBest val acc: {best_val_acc:.4f}")

### Compare Run 1 vs Run 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, 6)

axes[0].plot(epochs, history_v1['val_acc'], label='v1 (no correction)', linestyle='--')
axes[0].plot(epochs, history_v2['val_acc'], label='v2 (class weights)')
axes[0].axhline(majority_baseline, linestyle=':', color='red', label=f'{majority_class}-always baseline ({majority_baseline:.0%})')
axes[0].set_title('Val accuracy: v1 vs v2')
axes[0].set_xlabel('Epoch')
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].plot(epochs, history_v1['val_loss'], label='v1 (no correction)', linestyle='--')
axes[1].plot(epochs, history_v2['val_loss'], label='v2 (class weights)')
axes[1].set_title('Val loss: v1 vs v2')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle('Run 1 vs Run 2 comparison', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR + '/comparison.png', dpi=150)
plt.show()

In [ ]:
model_v2.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        preds = model_v2(images).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Confusion matrix — Run 2 (class weights)')
plt.tight_layout()
plt.savefig(RESULTS_DIR + '/v2_confusion_matrix.png', dpi=150)
plt.show()

## What to look for

After both runs, ask yourself:

- Did Run 1 val accuracy hover around 72%? That's the model doing nothing useful.
- Does Run 2's confusion matrix have a brighter diagonal? That means minority classes are being predicted, not ignored.
- Does Run 2's accuracy drop below Run 1? That's fine and expected — overall accuracy can go *down* when you fix imbalance, because the model stops cheating. What improves is per-class recall on the minority classes.
- Look at `recall` in the classification report for `df`, `vasc`, `akiec` — those are the hardest classes. Did they go from 0.00 to something real?